<a href="https://colab.research.google.com/github/diegoo7am/Curso-Intro-Python/blob/main/Modulo_02_Aplicaciones_Financieras/2_5_carteras_montecarlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[⬅ Volver al índice](https://colab.research.google.com/github/diegoo7am/Curso-Intro-Python/blob/main/indice.ipynb)

# Carteras de inversión y simulación de Monte Carlo

Hasta ahora hemos analizado los activos **de uno en uno**: su rentabilidad, su volatilidad y estrategias para operar con ellos. Pero un inversor casi nunca tiene un solo activo: tiene una **cartera**. En este notebook vamos a ver:

1. Cómo se **relacionan** los activos entre sí (correlación).
2. Cómo calcular la **rentabilidad y el riesgo de una cartera** a partir de los activos que la forman y sus pesos.
3. Cómo generar y comparar **muchas carteras a la vez**.
4. Cómo **simular** el comportamiento futuro de una cartera con Monte Carlo (lo que vimos en el 2.1, ahora con datos reales).
5. Cómo medir el riesgo de pérdidas extremas con el **VaR** y el **CVaR**.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

## Descarga de los datos

Trabajaremos con 4 fondos cotizados (ETF) que representan tipos de activo muy distintos:

| Ticker | Qué representa |
|---|---|
| SPY | Acciones de EE. UU. (S&P 500) |
| QQQ | Acciones tecnológicas de EE. UU. (Nasdaq 100) |
| TLT | Bonos del Tesoro de EE. UU. a largo plazo |
| GLD | Oro |

In [ ]:
activos = ["SPY", "QQQ", "TLT", "GLD"]

# auto_adjust=False: descarga también la columna 'Adj Close' (precio ajustado por dividendos y splits)
precios = yf.download(activos, start="2019-01-01", end="2024-12-31", auto_adjust=False)["Adj Close"]
precios = precios[activos]   # yfinance ordena las columnas alfabéticamente: las ponemos en nuestro orden

# Rentabilidades logarítmicas diarias, como en el 2.3
rent = np.log(precios).diff().dropna()
rent.head()

## 1. Correlación entre activos

La **correlación** mide si dos activos tienden a moverse a la vez. Se calcula a partir de la covarianza, dividiendo por las volatilidades para que el resultado quede siempre entre $-1$ y $1$:

$$\rho_{A,B} = \frac{\text{Cov}(r_A, r_B)}{\sigma_A \times \sigma_B}$$

* $\rho$ cercano a **1**: cuando uno sube, el otro también suele subir.
* $\rho$ cercano a **0**: sus movimientos no tienen relación.
* $\rho$ cercano a **-1**: cuando uno sube, el otro suele bajar.

Con pandas, `.corr()` calcula la correlación de **todas las parejas de columnas** de golpe y devuelve la **matriz de correlación**. Fíjate en que la calculamos sobre las **rentabilidades**, no sobre los precios.

Ref.:
* Función **corr**: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html

In [ ]:
correlaciones = rent.corr()
correlaciones.round(2)

La diagonal siempre vale 1 (cada activo está perfectamente correlacionado consigo mismo) y la matriz es simétrica: $\rho_{A,B} = \rho_{B,A}$.

Con muchos activos la tabla se lee mal, así que es habitual dibujarla como un **mapa de calor**, igual que hicimos en el 2.4 con los Sharpe de las medias móviles:

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(correlaciones.values, cmap="RdYlGn_r", vmin=-1, vmax=1)
plt.colorbar(label="Correlación")
plt.xticks(range(len(activos)), activos)
plt.yticks(range(len(activos)), activos)

# Escribimos el valor en el centro de cada casilla
for i in range(len(activos)):
    for j in range(len(activos)):
        plt.text(j, i, f"{correlaciones.iloc[i, j]:.2f}", ha="center", va="center")

plt.title("Matriz de correlación de las rentabilidades diarias")
plt.show()

Veamos qué significa esto con un gráfico de dispersión: cada punto es un día, con la rentabilidad de un activo en el eje X y la del otro en el eje Y.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(rent["SPY"], rent["QQQ"], s=5, alpha=0.5)
axes[0].set_title(f"SPY vs QQQ (ρ = {correlaciones.loc['SPY', 'QQQ']:.2f})")
axes[0].set_xlabel("Rentabilidad SPY")
axes[0].set_ylabel("Rentabilidad QQQ")
axes[0].grid(True)

axes[1].scatter(rent["SPY"], rent["TLT"], s=5, alpha=0.5, color="tab:orange")
axes[1].set_title(f"SPY vs TLT (ρ = {correlaciones.loc['SPY', 'TLT']:.2f})")
axes[1].set_xlabel("Rentabilidad SPY")
axes[1].set_ylabel("Rentabilidad TLT")
axes[1].grid(True)

plt.show()

**¿Por qué nos importa?** Si combinamos activos muy correlacionados (SPY y QQQ), cuando uno cae el otro también cae: la cartera no reduce el riesgo. Si combinamos activos poco correlacionados (SPY y TLT), las caídas de uno se compensan en parte con el otro. Esto es la **diversificación**, y lo vamos a medir en la sección siguiente.

<div style="background-color: #e0f7fa; padding: 10px;">

**EJERCICIO 1** Las correlaciones no son constantes en el tiempo.

- Calcula la matriz de correlación usando solo los datos de **2022** (año de fuertes subidas de tipos de interés).
- Compárala con la de todo el periodo. ¿Qué ha pasado con la correlación entre acciones (SPY) y bonos (TLT)? ¿Qué implica eso para un inversor que usaba los bonos para diversificar?

</div>

In [ ]:
# Escribe aquí tu código

## 2. Rentabilidad de una cartera

Una cartera se define por los **pesos** $w_i$: qué parte del dinero invertimos en cada activo. Los pesos suman 1. Por ejemplo, la clásica cartera **60/40**: 60 % en acciones (SPY) y 40 % en bonos (TLT).

La rentabilidad de la cartera cada día es la media ponderada de las rentabilidades de sus activos:

$$r_{p,t} = w_1 r_{1,t} + w_2 r_{2,t} + \dots + w_N r_{N,t} = \sum_{i=1}^{N} w_i \, r_{i,t}$$

Esa suma de productos es justo lo que hace el producto de matrices `@` que vimos en el 2.1 con las tiendas y los bolígrafos. Con todos los días a la vez:

$$\underbrace{\text{rent}}_{(\text{días},\ 4)} \ @ \ \underbrace{w}_{(4,)} \rightarrow \underbrace{r_p}_{(\text{días},)}$$

*Nota*: esta suma ponderada es exacta con rentabilidades aritméticas. Con rentabilidades logarítmicas diarias la diferencia es muy pequeña, así que la usaremos por sencillez.

In [ ]:
# Los pesos van en el mismo orden que las columnas de rent: SPY, QQQ, TLT, GLD
w_6040 = np.array([0.6, 0.0, 0.4, 0.0])   # shape (4,)
print(f"Suma de los pesos: {w_6040.sum()}")

# (días, 4) @ (4,) -> (días,): una rentabilidad por día
rent_6040 = rent @ w_6040
print(rent.shape, "@", w_6040.shape, "->", rent_6040.shape)
rent_6040.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(rent["SPY"].cumsum(), label="100 % SPY")
plt.plot(rent["TLT"].cumsum(), label="100 % TLT")
plt.plot(rent_6040.cumsum(), label="Cartera 60/40")
plt.title("Rentabilidad acumulada (logarítmica)")
plt.grid(True)
plt.legend()
plt.show()

Para comparar carteras usamos las métricas que ya conocemos: rentabilidad y volatilidad anualizadas (2.3) y ratio de Sharpe (2.4):

$$\mu_{anual} = \bar{r} \times 252 \qquad \sigma_{anual} = \sigma_r \times \sqrt{252} \qquad Sharpe = \frac{\mu_{anual}}{\sigma_{anual}}$$

In [ ]:
def calcular_metricas(rent: pd.Series) -> pd.Series:
    """Calcula la rentabilidad y la volatilidad anualizadas y el ratio de Sharpe.

    Args:
        rent (pd.Series): Rentabilidades logarítmicas diarias.

    Returns:
        pd.Series: Rentabilidad anual, volatilidad anual y Sharpe.
    """
    rent_anual = rent.mean() * 252
    vol_anual = rent.std() * np.sqrt(252)
    return pd.Series({
        "Rentabilidad anual": rent_anual,
        "Volatilidad anual": vol_anual,
        "Sharpe": rent_anual / vol_anual,
    })


tabla = pd.DataFrame({
    "100 % SPY": calcular_metricas(rent["SPY"]),
    "100 % TLT": calcular_metricas(rent["TLT"]),
    "60/40": calcular_metricas(rent_6040),
})
tabla.round(3)

### El efecto de la diversificación

Si la volatilidad de la cartera fuera simplemente la media ponderada de las volatilidades de sus activos, la 60/40 tendría:

$$0.6 \times \sigma_{SPY} + 0.4 \times \sigma_{TLT}$$

Pero la volatilidad real de la cartera es **menor**, porque SPY y TLT no se mueven a la vez: parte de sus oscilaciones se cancelan. Cuanto menor es la correlación, mayor es este efecto.

In [ ]:
vol_activos = rent.std() * np.sqrt(252)   # volatilidad anual de cada activo
vol_media_ponderada = vol_activos @ w_6040  # media ponderada de las volatilidades
vol_cartera = rent_6040.std() * np.sqrt(252)

print(f"Media ponderada de las volatilidades: {vol_media_ponderada:.2%}")
print(f"Volatilidad real de la cartera 60/40: {vol_cartera:.2%}")
print(f"Reducción de riesgo por diversificar:  {vol_media_ponderada - vol_cartera:.2%}")

<div style="background-color: #e0f7fa; padding: 10px;">

**EJERCICIO 2** Construye una cartera **equiponderada** (el mismo peso en los 4 activos).

- Calcula su rentabilidad diaria y añádela como nueva columna a la `tabla` usando `calcular_metricas`.
- ¿Tiene mejor Sharpe que la 60/40? ¿Y que el 100 % SPY?

</div>

In [ ]:
# Escribe aquí tu código

## 3. Muchas carteras a la vez

¿Y si en lugar de elegir los pesos a mano generamos **muchas carteras con pesos aleatorios** para ver qué combinaciones de rentabilidad y riesgo son posibles con estos 4 activos?

Para que los pesos de cada cartera sumen 1:
1. Generamos números aleatorios entre 0 y 1 con `np.random.random`: una fila por cartera y una columna por activo.
2. Dividimos cada fila entre su suma. Como vimos en el 2.1 con el *broadcasting*, la suma de cada fila tiene que ser una **columna** de shape `(n_carteras, 1)`.

$$w_i = \frac{x_i}{\sum_{j} x_j}$$

In [ ]:
np.random.seed(42)   # fija los números aleatorios para que el resultado sea siempre el mismo
n_carteras = 1000

aleatorios = np.random.random(size=(n_carteras, len(activos)))   # shape (1000, 4)
suma_filas = aleatorios.sum(axis=1).reshape(-1, 1)               # shape (1000, 1): una suma por cartera
pesos = aleatorios / suma_filas                                  # (1000, 4) / (1000, 1) -> (1000, 4)

print(pesos.shape)
print(pesos[:3].round(3))                   # pesos de las 3 primeras carteras
print(pesos.sum(axis=1)[:3])                # comprobamos que cada fila suma 1

Ahora calculamos las rentabilidades diarias de **las 1000 carteras a la vez** con un único producto de matrices. Necesitamos que los activos estén en las filas de la matriz de pesos, así que la trasponemos con `.T`:

$$\underbrace{\text{rent}}_{(\text{días},\ 4)} \ @ \ \underbrace{\text{pesos.T}}_{(4,\ 1000)} \rightarrow \underbrace{\text{rent\_carteras}}_{(\text{días},\ 1000)}$$

Cada **columna** del resultado es la serie de rentabilidades de una cartera, así que calculamos las medias y desviaciones típicas por columnas (`axis=0`).

In [ ]:
rent_carteras = rent.values @ pesos.T            # (días, 4) @ (4, 1000) -> (días, 1000)
print(rent_carteras.shape)

# axis=0: un resultado por columna (por cartera). ddof=1 para que coincida con .std() de pandas
mu_carteras = rent_carteras.mean(axis=0) * 252
vol_carteras = rent_carteras.std(axis=0, ddof=1) * np.sqrt(252)
sharpe_carteras = mu_carteras / vol_carteras

print(mu_carteras.shape, vol_carteras.shape, sharpe_carteras.shape)

Dibujamos cada cartera como un punto en el plano **riesgo (X) / rentabilidad (Y)**, con el color según su Sharpe, y marcamos también los activos individuales.

In [ ]:
mu_activos = rent.mean() * 252

plt.figure(figsize=(10, 6))
plt.scatter(vol_carteras, mu_carteras, c=sharpe_carteras, cmap="viridis", s=10)
plt.colorbar(label="Sharpe")

# Activos individuales
plt.scatter(vol_activos, mu_activos, color="red", marker="X", s=100, label="Activos individuales")
for activo in activos:
    plt.annotate(activo, (vol_activos[activo], mu_activos[activo]), xytext=(5, 5), textcoords="offset points")

plt.xlabel("Volatilidad anual")
plt.ylabel("Rentabilidad anual")
plt.title(f"{n_carteras} carteras aleatorias con SPY, QQQ, TLT y GLD")
plt.grid(True)
plt.legend()
plt.show()

Fíjate en que muchas carteras tienen **menos volatilidad que cualquiera de los cuatro activos por separado**, y aun así una rentabilidad intermedia. Ninguna inversión individual consigue esa combinación: es el efecto de la diversificación que vimos antes, aplicado a 1000 carteras.

<div style="background-color: #e0f7fa; padding: 10px;">

**EJERCICIO 3** Repite la nube de carteras usando **solo SPY y QQQ** (1000 carteras de 2 activos).

- Genera los pesos aleatorios, calcula la rentabilidad y la volatilidad anual de cada cartera y dibújalas junto con los dos activos.
- ¿Qué forma tiene ahora la nube? ¿Por qué apenas se reduce el riesgo al combinar estos dos activos?

</div>

In [ ]:
# Escribe aquí tu código

## 4. Simulación de Monte Carlo de una cartera

Todo lo anterior mira al **pasado**. ¿Qué podría pasarle a nuestra cartera 60/40 en el **próximo año**?

En el 2.1 simulamos los ingresos de una empresa con una distribución normal cuya media y desviación típica nos daban en el enunciado. Ahora haremos lo mismo con la rentabilidad diaria de la cartera, pero **estimando la media y la desviación típica a partir de los datos históricos**:

$$r_t \sim N(\bar{r}, \sigma_r)$$

Si invertimos un capital $V_0$, como las rentabilidades son logarítmicas, el valor al cabo de $T$ días es:

$$V_T = V_0 \times e^{\sum_{t=1}^{T} r_t}$$

In [ ]:
mu_diaria = rent_6040.mean()
vol_diaria = rent_6040.std()
print(f"Rentabilidad media diaria:  {mu_diaria:.5f}  (anual: {mu_diaria * 252:.2%})")
print(f"Volatilidad diaria:         {vol_diaria:.5f}  (anual: {vol_diaria * np.sqrt(252):.2%})")

In [ ]:
np.random.seed(42)
n_escenarios = 10000
n_dias = 252          # un año de sesiones
capital = 10000       # euros invertidos hoy

# Una fila por escenario y una columna por día, como en el 2.1
r_sim = np.random.normal(loc=mu_diaria, scale=vol_diaria, size=(n_escenarios, n_dias))   # (10000, 252)

# cumsum(axis=1): rentabilidad acumulada día a día dentro de cada escenario (cada fila)
valor = capital * np.exp(r_sim.cumsum(axis=1))   # (10000, 252): valor de la cartera cada día
valor_final = valor[:, -1]                      # (10000,): última columna = valor al cabo de un año

In [ ]:
plt.figure(figsize=(12, 5))
# valor[:100].T -> (252, 100): plt.plot dibuja una línea por columna, es decir, 100 escenarios
plt.plot(valor[:100].T, alpha=0.4, linewidth=0.8)
plt.axhline(capital, color="black", linestyle="--", label="Capital inicial")
plt.title("100 escenarios simulados de la cartera 60/40")
plt.xlabel("Días")
plt.ylabel("Valor de la cartera (€)")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
p5, p50, p95 = np.percentile(valor_final, [5, 50, 95])

print(f"Valor medio al cabo de un año:  {valor_final.mean():,.0f} €")
print(f"Percentil 5 %:                  {p5:,.0f} €")
print(f"Mediana (percentil 50 %):       {p50:,.0f} €")
print(f"Percentil 95 %:                 {p95:,.0f} €")
# valor_final < capital da True/False; la media de True/False es la proporción de True
print(f"Probabilidad de perder dinero:  {np.mean(valor_final < capital):.2%}")

plt.figure(figsize=(10, 5))
plt.hist(valor_final, bins=60)
plt.axvline(capital, color="black", linestyle="--", label="Capital inicial")
plt.axvline(p5, color="red", linestyle="--", label="Percentil 5 %")
plt.axvline(p95, color="green", linestyle="--", label="Percentil 95 %")
plt.title("Distribución del valor de la cartera dentro de un año")
plt.xlabel("Valor (€)")
plt.ylabel("Frecuencia")
plt.grid(True)
plt.legend()
plt.show()

**Ojo con la hipótesis.** Hemos supuesto que las rentabilidades diarias son normales y que la media y la volatilidad del futuro serán las del pasado. En el 2.3 vimos que las rentabilidades reales tienen **colas más gruesas** que la normal (los días extremos son más frecuentes), así que esta simulación probablemente **infravalora las pérdidas extremas**. Un modelo es tan bueno como sus hipótesis.

## 5. Medir el riesgo de pérdidas: VaR y CVaR

La volatilidad mide cuánto oscila la cartera, tanto hacia arriba como hacia abajo. Pero a un inversor le preocupa sobre todo **cuánto puede perder**. Para eso se usan dos medidas:

* **VaR (*Value at Risk*) al 95 %**: la pérdida que **solo se supera en el 5 % de los escenarios**. Es la distancia entre el capital inicial y el percentil 5 % del valor final.

$$VaR_{95\%} = V_0 - \text{percentil}_{5\%}(V_T)$$

* **CVaR (*Conditional VaR*) al 95 %**: la **pérdida media en ese 5 % de peores escenarios**. Responde a la pregunta: "si las cosas van mal, ¿cuánto pierdo de media?".

$$CVaR_{95\%} = V_0 - \text{media}\left(V_T \ \middle|\ V_T \le \text{percentil}_{5\%}(V_T)\right)$$

El CVaR siempre es mayor o igual que el VaR, porque tiene en cuenta lo malos que son los peores escenarios y no solo dónde empiezan.

In [ ]:
peores = valor_final[valor_final <= p5]   # filtramos el 5 % de peores escenarios
var_95 = capital - p5
cvar_95 = capital - peores.mean()

print(f"Escenarios en el 5 % peor: {len(peores)} de {len(valor_final)}")
print(f"VaR 95 %:  {var_95:,.0f} €  ({var_95 / capital:.2%} del capital)")
print(f"CVaR 95 %: {cvar_95:,.0f} €  ({cvar_95 / capital:.2%} del capital)")

intervalos = np.histogram_bin_edges(valor_final, bins=60)   # mismos intervalos para los dos histogramas

plt.figure(figsize=(10, 5))
plt.hist(valor_final, bins=intervalos, color="lightgray")
plt.hist(peores, bins=intervalos, color="red", label="5 % peores escenarios")
plt.axvline(p5, color="red", linestyle="--", label=f"Percentil 5 % → VaR = {var_95:,.0f} €")
plt.axvline(peores.mean(), color="darkred", linestyle="-", label=f"Media del 5 % peor → CVaR = {cvar_95:,.0f} €")
plt.axvline(capital, color="black", linestyle="--", label="Capital inicial")
plt.title("VaR y CVaR al 95 % de la cartera 60/40 (horizonte 1 año)")
plt.xlabel("Valor (€)")
plt.ylabel("Frecuencia")
plt.grid(True)
plt.legend()
plt.show()

<div style="background-color: #e0f7fa; padding: 10px;">

**EJERCICIO 4** ¿Cuánto riesgo de pérdida añade invertirlo todo en acciones?

- Repite la simulación de Monte Carlo (10.000 escenarios, 1 año, 10.000 €) para una cartera **100 % SPY**. Consejo: mete la simulación en una **función** que reciba las rentabilidades diarias de una cartera y devuelva los valores finales, y úsala para las dos carteras.
- Construye una tabla que compare la 60/40 y el 100 % SPY: valor medio, probabilidad de perder dinero, VaR 95 % y CVaR 95 %.
- ¿Compensa la mayor rentabilidad esperada de SPY el riesgo extra?

</div>

In [ ]:
# Escribe aquí tu código

## Resumen

* La **correlación** (`.corr()`) mide si los activos se mueven a la vez, y cambia con el tiempo.
* La rentabilidad diaria de una cartera es `rentabilidades @ pesos`; con `@` y la matriz de pesos traspuesta podemos calcular **muchas carteras a la vez**.
* Combinar activos poco correlacionados **reduce el riesgo** sin renunciar a toda la rentabilidad: diversificación.
* Con **Monte Carlo** simulamos miles de futuros posibles a partir de la media y la volatilidad históricas, y con el **VaR** y el **CVaR** medimos las pérdidas en los peores escenarios.